In [ ]:
import pandas as pd
import numpy as np
import torch
import plotly.graph_objects as go
import matplotlib.pyplot as plt

from src.utils import generate_mask_tensor
from src.embedding import embed
from src.gp_ccm import GP_ccm_sig, run_sigGPCCM_experiment, GP_ccm_sig_predict
from src.sp_ccm import run_SP_CCM, SP_CCM_iaaft, run_ccm_experiment
from src.iaaft import surrogates

from scipy.stats import ranksums
torch.set_printoptions(sci_mode = False)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print()

# Load data

In [ ]:
co2_norm = torch.load("data/CO2_vostok_stan_400kyr_timeseries.pt").to(torch.float32)
temp_norm = torch.load("data/TEMP_vostok_stan_400kyr_timeseries.pt").to(torch.float32)

In [ ]:
k = 3
N_TRAIN = torch.tensor([200]).to(device)

##############
### sigCCM ###
##############

sig_filter = torch.ones(size = (k, )).to(device)
sig_shift = torch.tensor(sig_filter.shape[0] - 1).to(device) + 2

NOISE_SCALE = torch.tensor([0.05], device = device)
RBF_SCALE = torch.tensor([0.3], device = device)

y_embeddings, x_gt = embed(
    filter = sig_filter, 
    y = co2_norm, 
    x = temp_norm, 
    max_pos_offset = sig_shift, 
    device = device)

rho, nlml, x_test_mean_est, x_test_covar_est = GP_ccm_sig_predict(
    y_embeddings_train = y_embeddings[0 : N_TRAIN].unsqueeze(-1), # [N, E, 1]
    y_embeddings_test = y_embeddings[N_TRAIN : ].unsqueeze(-1), 
    x_train = x_gt[0 : N_TRAIN], # [N]
    x_test = x_gt[N_TRAIN : ], 
    noise = NOISE_SCALE, 
    rbf_sigma = RBF_SCALE, 
    device = device)

print(rho)

In [ ]:
k = 4
N_TRAIN = torch.tensor([200]).to(device)

##############
### sigCCM ###
##############

sig_filter = torch.ones(size = (k, )).to(device)
sig_shift = torch.tensor(sig_filter.shape[0] - 1).to(device) + 4

NOISE_SCALE = torch.tensor([0.05], device = device)
RBF_SCALE = torch.tensor([0.3], device = device)

y_embeddings, x_gt = embed(
    filter = sig_filter, 
    y = co2_norm, 
    x = temp_norm, 
    max_pos_offset = sig_shift, 
    device = device)

rho, nlml, x_test_mean_est, x_test_covar_est = GP_ccm_sig_predict(
    y_embeddings_train = y_embeddings[0 : N_TRAIN].unsqueeze(-1), # [N, E, 1]
    y_embeddings_test = y_embeddings[N_TRAIN : ].unsqueeze(-1), 
    x_train = x_gt[0 : N_TRAIN], # [N]
    x_test = x_gt[N_TRAIN : ], 
    noise = NOISE_SCALE, 
    rbf_sigma = RBF_SCALE, 
    device = device)

print(rho)

In [ ]:
x_test_covar_est.cpu()

In [ ]:
# Initalise
fig = go.Figure()

fig.add_trace(go.Scatter(x = torch.arange(0, co2_norm.shape[0]), y = temp_norm, # reversing the meaning of x
                        mode = 'lines',
                        name = 'temperature',
                        line_color = "black"))

fig.add_trace(go.Scatter(x = torch.arange(N_TRAIN.item(), co2_norm.shape[0]), y = temp_norm, # reversing the meaning of x
                        mode = 'lines',
                        name = 'reconstruction',
                        line_color = "blue"))

for i in range(40):
    # Sample via Cholesky
    L = torch.linalg.cholesky(x_test_covar_est.cpu())
    Z = torch.randn(size = [L.shape[-1]], dtype = torch.double).unsqueeze(0)
    sample = x_test_mean_est.cpu() + torch.matmul(Z, L)  
    fig.add_trace(go.Scatter(x = np.arange(start = -39, stop = (0 + 1)), y = sample.squeeze(), 
                        mode = 'lines',
                        name = 'Sample',
                        showlegend = False,
                        line = dict(
                                color='#0061ff',
                                width = 0.5
                            )
                        ))

fig.update_layout(template = "simple_white")
fig.update_layout(font_family = "Lato")
fig.update_layout(legend = dict(x = 0.01, y = 0.9, bgcolor = "rgba(0,0,0,0)"))

fig.update_layout(width = 1000, height = 500)

fig.show()

# Ensure symmetry and pos

In [ ]:
x_test_covar_est

# CO2 -> temp

In [ ]:
k = 2
N_TRAIN = torch.tensor([200]).to(device)

##############
### sigCCM ###
##############

sig_filter = torch.ones(size = (k, )).to(device)
sig_shift = torch.tensor(sig_filter.shape[0] - 1).to(device) -1 
# -1 is towards overlap

NOISE_SCALE = torch.tensor([0.05], device = device)
RBF_SCALE = torch.tensor([0.3], device = device)

y_embeddings, x_gt = embed(
    filter = sig_filter, 
    y = temp_norm, 
    x = co2_norm, 
    max_pos_offset = sig_shift, 
    device = device)

rho, nlml, x_test_mean_est, x_test_covar_est = GP_ccm_sig_predict(
    y_embeddings_train = y_embeddings[0 : N_TRAIN].unsqueeze(-1), # [N, E, 1]
    y_embeddings_test = y_embeddings[N_TRAIN : ].unsqueeze(-1), 
    x_train = x_gt[0 : N_TRAIN], # [N]
    x_test = x_gt[N_TRAIN : ], 
    noise = NOISE_SCALE, 
    rbf_sigma = RBF_SCALE, 
    device = device)

print(rho)